# SAFE binary v5 — hard-case 2차 반복 재학습

소비된 blind v1·v2는 개발 회귀셋으로 사용하고, 새 blind v3는 후보 선택 후 정확히 한 번만 실행한다. blind 원문은 학습에 절대 병합하지 않는다.

In [1]:
# 1. A100 및 저장소 확인
import subprocess, sys, json, hashlib, re, shutil
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다.'
print(torch.cuda.get_device_name(0))
REPO=Path('/content/thisabled-ai')
BRANCH='feature/grooming-augmentation'
REMOTE='https://github.com/threeGuineas/thisabled-ai.git'
if REPO.exists():
    subprocess.run(['git','pull','--ff-only','origin',BRANCH],cwd=REPO,check=True)
else:
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REMOTE,str(REPO)],check=True)
assert (REPO/'.git').exists(),f'저장소 준비 실패: {REPO}'
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],cwd=REPO,check=True)
sys.path.insert(0,str(REPO))


In [2]:
# 2. data_bundle.zip 업로드 — VS Code/Jupyter widget
import io, zipfile, ipywidgets as widgets
from IPython.display import display
uploader=widgets.FileUpload(accept='.zip',multiple=False,description='data_bundle.zip 선택')
def on_upload(change):
    value=uploader.value
    if not value:return
    item=next(iter(value.values())) if isinstance(value,dict) else value[0]
    with zipfile.ZipFile(io.BytesIO(bytes(item['content']))) as z:
        required={'data/eval/aihub_train.jsonl','data/eval/aihub_real_holdout.jsonl','data/eval/beep_real_holdout.jsonl'}
        missing=sorted(required-set(z.namelist())); assert not missing,missing
        z.extractall(REPO)
    print('업로드 완료')
uploader.observe(on_upload,names='value'); display(uploader)


In [3]:
# 3. 데이터 빌드 및 누수 가드
required=[REPO/'data/eval/aihub_train.jsonl',REPO/'data/eval/aihub_real_holdout.jsonl',REPO/'data/eval/beep_real_holdout.jsonl']
assert all(p.exists() for p in required),[str(p) for p in required if not p.exists()]
steps=[[sys.executable,'scripts/download_seed_datasets.py'],[sys.executable,'scripts/build_processed_dataset.py'],[sys.executable,'scripts/build_final_dataset.py','--synth-repeat','1','--include-aihub-train'],[sys.executable,'scripts/build_safe_hardcase_dataset.py','--include-v2','--output','data/synthetic/safe_hardcases_v2/train.jsonl','--forbidden','tests/fixtures/safe_blind_v2.jsonl','--forbidden','tests/fixtures/safe_blind_v3.jsonl']]
for cmd in steps: subprocess.run(cmd,cwd=REPO,check=True)
import pandas as pd
train=pd.read_parquet(REPO/'data/processed/train.parquet')
hard=[json.loads(x) for x in (REPO/'data/synthetic/safe_hardcases_v2/train.jsonl').read_text().splitlines() if x]
def norm(x):return re.sub(r'[^0-9a-z가-힣]+','',str(x).lower())
for name in ['safe_blind_v1.jsonl','safe_blind_v2.jsonl','safe_blind_v3.jsonl']:
    blind=[json.loads(x) for x in (REPO/'tests/fixtures'/name).read_text().splitlines() if x]
    assert not ({norm(x['text']) for x in blind}&{norm(x['text']) for x in hard})
print('base train',len(train),'hardcases',len(hard),'labels',pd.Series([x['label'] for x in hard]).value_counts().to_dict())


In [4]:
# 4. 개발 평가 함수 — blind v3는 여기서 읽지 않음
import numpy as np, yaml
from sklearn.metrics import confusion_matrix, f1_score
from transformers import AutoTokenizer,AutoModelForSequenceClassification
from serving.safety_server.app import _rule_hit
real=[]
for name in ['aihub_real_holdout.jsonl','beep_real_holdout.jsonl']:
    real += [json.loads(x) for x in (REPO/'data/eval'/name).read_text().splitlines() if x]
dev=[]
for name in ['safe_blind_v1.jsonl','safe_blind_v2.jsonl']: dev += [json.loads(x) for x in (REPO/'tests/fixtures'/name).read_text().splitlines() if x]
groom=[]
for split in ['val','test']: groom += [json.loads(x) for x in (REPO/f'data/synthetic/emergency/3a/{split}.jsonl').read_text().splitlines() if x]
def load_predict(path):
    tok=AutoTokenizer.from_pretrained(path); model=AutoModelForSequenceClassification.from_pretrained(path).cuda().eval(); assert model.config.num_labels==2
    def predict(texts,batch=128):
        out=[]
        with torch.inference_mode():
            for i in range(0,len(texts),batch):
                enc=tok(texts[i:i+batch],padding=True,truncation=True,max_length=128,return_tensors='pt').to('cuda')
                out.extend(torch.softmax(model(**enc).logits,dim=-1)[:,1].cpu().tolist())
        return np.array(out)
    return model,predict
def evaluate_dev(path):
    model,predict=load_predict(path)
    yr=np.array([int(int(x['label'])>0) for x in real]); pr=predict([x['text'] for x in real]); rr=np.array([_rule_hit(x['text']) for x in real])
    yd=np.array([x['label'] for x in dev]); pdv=predict([x['text'] for x in dev]); rd=np.array([_rule_hit(x['text']) for x in dev])
    pg=predict([x['text'] for x in groom])
    candidates=[]
    for adult in np.arange(.40,.86,.01):
        minor=max(.35,round(float(adult)-.16,2))
        rpred=(pr>=adult)|rr
        dthr=np.array([minor if x['receiver_is_minor'] else adult for x in dev]); dpred=(pdv>=dthr)|rd
        tn,fp,fn,tp=confusion_matrix(yr,rpred,labels=[0,1]).ravel(); dtn,dfp,dfn,dtp=confusion_matrix(yd,dpred,labels=[0,1]).ravel()
        fraud=[i for i,x in enumerate(dev) if x['slice']=='fraud_credentials']; grooming=[i for i,x in enumerate(dev) if x['slice'].startswith('grooming')]; ext=[i for i,x in enumerate(dev) if x['slice']=='digital_extortion']
        row={'adult':round(float(adult),2),'minor':minor,'real_recall':tp/(tp+fn),'real_specificity':tn/(tn+fp),'dev_recall':dtp/(dtp+dfn),'dev_specificity':dtn/(dtn+dfp),'fraud_recall':float(dpred[fraud].mean()),'grooming_recall':float(dpred[grooming].mean()),'extortion_recall':float(dpred[ext].mean()),'synthetic_grooming_recall':float((pg>=minor).mean())}
        row['pass']=all([row['real_recall']>=.80,row['real_specificity']>=.80,row['dev_recall']>=.85,row['dev_specificity']>=.90,row['fraud_recall']>=.80,row['grooming_recall']>=.80,row['extortion_recall']>=.80])
        candidates.append(row)
    passing=[x for x in candidates if x['pass']]; best=max(passing,key=lambda x:(x['real_specificity'],x['dev_specificity'])) if passing else max(candidates,key=lambda x:(min(x['dev_recall'],x['dev_specificity'],x['fraud_recall']),x['real_specificity']))
    del model; torch.cuda.empty_cache(); return best


In [5]:
# 5. v2 repeat 1→2→3 재학습. 개발 게이트 통과 시 즉시 중단
ATTEMPTS=[]; SELECTED=None
base=yaml.safe_load((REPO/'configs/module1_binary_hardcases_v2.yaml').read_text())
for repeat in [1,2,3]:
    cfg=json.loads(json.dumps(base)); name=f'module1_binary_hardcases_v2_r{repeat}'; cfg['data']['extra_train_repeat']=repeat; cfg['model']['checkpoint_dir']=f'models/checkpoints/{name}'; cfg['paths']['checkpoint_dir']=f'models/checkpoints/{name}'
    temp=Path(f'/content/{name}.yaml'); temp.write_text(yaml.safe_dump(cfg,allow_unicode=True,sort_keys=False))
    subprocess.run([sys.executable,'scripts/train_module1.py','--config',str(temp)],cwd=REPO,check=True)
    ckpt=REPO/cfg['model']['checkpoint_dir']; result=evaluate_dev(ckpt); result.update({'repeat':repeat,'checkpoint':str(ckpt)}); ATTEMPTS.append(result); print(json.dumps(result,ensure_ascii=False,indent=2))
    if result['pass']: SELECTED=result; break
report=REPO/'reports/validation_reports/module1_binary_hardcases_v2/dev_attempts.json'; report.parent.mkdir(parents=True,exist_ok=True); report.write_text(json.dumps(ATTEMPTS,ensure_ascii=False,indent=2))
assert SELECTED is not None, '3회 모두 개발 게이트 실패 — blind v2 실행 및 HF 업로드 금지'
print('SELECTED',SELECTED)


In [6]:
# 6. 후보 고정 후 blind v3 최초 1회 평가
blind_path=REPO/'tests/fixtures/safe_blind_v3.jsonl'; before=hashlib.sha256(blind_path.read_bytes()).hexdigest()
out=REPO/'artifacts/safe_blind_v3_results.json'
subprocess.run([sys.executable,'scripts/evaluate_safe_blind.py','--model',SELECTED['checkpoint'],'--data',str(blind_path),'--adult-threshold',str(SELECTED['adult']),'--minor-threshold',str(SELECTED['minor']),'--output',str(out)],cwd=REPO,check=True)
assert hashlib.sha256(blind_path.read_bytes()).hexdigest()==before
BLIND=json.loads(out.read_text()); m=BLIND['overall']; risk_slices=['fraud_credentials','routine_recon','coercive_control','digital_extortion','grooming_threat']; slice_r={s:BLIND['by_slice'][s]['risk_recall'] for s in risk_slices}
BLIND_PASS=bool(m['risk_recall']>=.80 and m['specificity']>=.90 and min(slice_r.values())>=.75)
print({'blind_pass':BLIND_PASS,'overall':m,'risk_slices':slice_r,'sha256':before})
assert BLIND_PASS, 'blind v3 실패 — 업로드 금지, v3는 회귀셋으로 격하'


In [7]:
# 7. 명시적으로 켠 경우에만 HF 업로드
UPLOAD_TO_HF=False
if UPLOAD_TO_HF:
    from getpass import getpass
    from huggingface_hub import login,upload_folder
    token=getpass('HF write token: '); login(token=token,add_to_git_credential=False); del token
    url=upload_folder(repo_id='soyuncj/thisabled-safety-kcelectra',folder_path=SELECTED['checkpoint'],commit_message=f"retrain hardcases r{SELECTED['repeat']} blind-v2 approved",ignore_patterns=['checkpoint-*','optimizer*','scheduler*','trainer_state*','rng_state*','training_args*'])
    print('HF_COMMIT_URL:',url)
else: print('검증 완료. 업로드는 비활성 상태입니다.')
